# Parallelism Strategies for Large Models

> Collective communication specifies how GPUs exchange data; a parallelism strategy decides which dimensions of the model and training state to partition. A 100B-parameter model stored at 2 bytes per parameter needs 200 GB for weights alone, so it cannot fit on one 80 GB GPU.
>
> **Data dimension**: Data Parallelism replicates the model and partitions the batch, while ZeRO removes replicas of parameters, gradients, and optimizer states.
>
> **Model dimensions**: Tensor Parallelism partitions matrices within a layer, Pipeline Parallelism partitions layers, Sequence Parallelism partitions the sequence, and Expert Parallelism partitions MoE experts.
>
> A real training configuration usually combines several dimensions. The choice depends on memory capacity, link bandwidth, and load balance.

With eight GPUs the aggregate capacity may be sufficient, but we must still decide which weights and samples each GPU owns and when results are exchanged. We will begin with Data Parallelism and use one memory ledger to compare what each strategy saves and what communication it adds.


## 1. Data Parallelism

**Partitioned object**: training data.

Suppose eight GPUs process a batch of 64 samples. Data Parallelism (DP) gives each GPU eight samples while every GPU retains a complete model copy.

DDP follows four steps:

1. Each GPU runs forward and backward on its local samples to obtain local gradients.
2. An all-reduce averages gradients across GPUs.
3. Every GPU updates its parameters with the same averaged gradients.
4. Identical initial parameters, gradients, and update rules keep all replicas synchronized.

DP multiplies throughput by $N$, but saves no model memory. When a model no longer fits on one GPU, the model itself must be partitioned. ZeRO first removes redundant optimizer states, gradients, and parameters from DP while keeping communication volume in the same order of magnitude.


In [ ]:
# === Per-GPU memory for three ZeRO stages: 7B model across eight GPUs ===
# During training, one BF16 parameter uses about 16 bytes:
#   2 parameter + 2 gradient + 12 optimizer state (Adam m and v at four bytes each plus four-byte master weight)
P = 7e9        # parameter count
N = 8          # GPU count

ddp_per_card   = 16 * P                        # complete copy on every GPU
zero1_per_card = 2 * P + 2 * P + 12 * P / N    # shard optimizer state only
zero2_per_card = 2 * P + 2 * P / N + 12 * P / N  # shard optimizer and gradients
zero3_per_card = 16 * P / N                    # shard everything

print(f"{'Scheme':<14}{'Per GPU (GB)':>12}{'vs DDP':>12}")
print("-" * 40)
for name, val in [("DDP", ddp_per_card),
                  ("ZeRO-1", zero1_per_card),
                  ("ZeRO-2", zero2_per_card),
                  ("ZeRO-3", zero3_per_card)]:
    print(f"{name:<14}{val/1e9:>10.1f}   {val/ddp_per_card*100:>8.1f}%")

print()
print("Key observation: ZeRO-3 reduces per-GPU memory from 112 GB to 14 GB, exactly one eighth.")
print("The cost is repeated parameter all-gather and gradient reduce-scatter, increasing bandwidth sensitivity.")


### 1.1 The Three Memory Costs of Data Parallelism

Why did the code use 16 bytes per parameter? Training a BF16 model with Adam requires approximately:

- 2 bytes for the BF16 parameter;
- 2 bytes for its gradient;
- 12 bytes for Adam's first moment, second moment, and a high-precision master weight.

Backpropagation must retain gradients, while Adam must remember two historical statistics and use a stable high-precision copy for updates. This 16-byte rule is a useful estimate: a 7B model needs about $7	ext{B}	imes16=112$ GB of fixed training state.


## 2. Tensor Parallelism

DP replicates the model, and ZeRO communicates frequently. Tensor Parallelism (TP) instead divides every layer's weights into $N$ pieces.

For $y=xW$, with $W$ shaped $[d_{in},d_{out}]$, there are two natural partitions:

- **Column parallel** partitions the output dimension. Each GPU stores $[d_{in},d_{out}/N]$ and independently computes its output columns.
- **Row parallel** partitions the input dimension. Each GPU stores $[d_{in}/N,d_{out}]$ and computes a partial sum; an all-reduce is required to obtain the complete result.

Column partitions are independent because all GPUs receive the same input and own different output columns. Row partitions each contribute to every output column, so their partial sums must be added. A column-parallel result may still need communication if the next layer requires the complete output. Practical systems therefore combine both forms.


In [ ]:
# === First inspect two partition styles on a tiny matrix with two GPUs ===
import numpy as np
np.random.seed(0)

# A linear layer y=xW with four input and six output dimensions
d_in, d_out = 4, 6
W = np.random.randn(d_in, d_out)   # weights [4,6]

# Column partition: split output columns into two [4,3] shards
col_shards = np.split(W, 2, axis=1)
print("Column partition by output dimension; each GPU receives several columns:")
for r, s in enumerate(col_shards):
    print(f"  rank {r} weight shape: {s.shape}")

# Row partition: split input rows into two [2,6] shards
row_shards = np.split(W, 2, axis=0)
print("\nRow partition by input dimension; each GPU receives several rows:")
for r, s in enumerate(row_shards):
    print(f"  rank {r} weight shape: {s.shape}")

print()
print("Key observation: every partition preserves the total parameter count and changes only the split direction.")
print("Next, combine these two partitions into a complete MLP.")


### 2.1 Combining Column and Row Parallelism

An MLP contains an expansion matrix $W_1$, an element-wise activation, and a contraction matrix $W_2$. Megatron-LM partitions them in complementary directions:

- use column parallelism for $W_1$;
- apply SiLU or another element-wise activation locally;
- use row parallelism for $W_2$, followed by one all-reduce.

The columns produced by the first layer are exactly the input rows owned by the second layer. The activation does not mix elements, so no intermediate communication is necessary. The entire MLP needs only the final all-reduce. The next example verifies this with concrete numbers.


In [ ]:
# === Hand verification: a two-GPU TP MLP needs only one all-reduce ===
import numpy as np
np.random.seed(0)

d_model = 4     # input dimension
hidden  = 6     # intermediate dimension
world_size = 2

W1 = np.random.randn(d_model, hidden)   # first layer [4,6]
W2 = np.random.randn(hidden, d_model)   # second layer [6,4]
x  = np.random.randn(1, d_model)        # input [1,4]


def silu(z):
    """Elementwise SiLU activation, computed independently on every GPU."""
    return z / (1.0 + np.exp(-z))

# Reference: full computation on one GPU
y_ref = silu(x @ W1) @ W2

# TP: split W1 by columns and W2 by rows
W1_shards = np.split(W1, world_size, axis=1)   # each [4,3]
W2_shards = np.split(W2, world_size, axis=0)   # each [3,4]

partials = []
for r in range(world_size):
    h_local = silu(x @ W1_shards[r])   # each GPU computes its intermediate columns without communication
    y_partial = h_local @ W2_shards[r] # each GPU computes a partial sum
    partials.append(y_partial)
    print(f"rank {r}: local output shape {y_partial.shape}, values {y_partial.ravel()}")

# Simulate one all-reduce sum
y_tp = sum(partials)
print()
print(f"Reference y_ref: {y_ref.ravel()}")
print(f"TP reconstructed y_tp: {y_tp.ravel()}")
print(f"Maximum error: {np.abs(y_ref - y_tp).max():.2e}")

print()
print("Key observation: the column, activation, and row path needs only one all-reduce at its end.")
print("This is TP's core: the communication count is fixed and does not double with layer count.")


### 2.2 Partitioning Attention Heads

Attention has a natural partition boundary: each head computes dot products only with its own Q, K, and V. With 32 heads and four GPUs, each GPU can independently process eight heads without communication inside attention.

As in the MLP, the QKV projection is column-parallel and the output projection is row-parallel, followed by one all-reduce. The number of heads must be divisible by the TP degree.


In [ ]:
# === Partition Attention by head: verify independent computation per GPU followed by summation ===
import numpy as np
np.random.seed(2)

d_model, num_heads = 8, 4
head_dim = d_model // num_heads    # two dimensions per head
seq_len = 3
world_size = 2                     # two GPUs, two heads per GPU

x = np.random.randn(1, seq_len, d_model)
# Independent weights per head, an MHA-equivalent form convenient for head partitioning
W_qkv = np.random.randn(num_heads, d_model, 3 * head_dim)
W_out = np.random.randn(num_heads, head_dim, d_model)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def head_forward(x, h):
    """Complete forward pass for one head: projection, dot product, softmax, output projection."""
    qkv = (x @ W_qkv[h]).reshape(1, seq_len, 3, head_dim)
    q, k, v = qkv[..., 0, :], qkv[..., 1, :], qkv[..., 2, :]
    sc = q @ k.transpose(0, 2, 1) / np.sqrt(head_dim)
    att = softmax(sc) @ v
    return att @ W_out[h]

# Reference: sum all heads, equivalent to full MHA
y_ref = sum(head_forward(x, h) for h in range(num_heads))

# TP: assign two heads to each GPU with no communication
heads_per_rank = num_heads // world_size
partials = []
for r in range(world_size):
    local = range(r * heads_per_rank, (r + 1) * heads_per_rank)
    partial = sum(head_forward(x, h) for h in local)
    partials.append(partial)

y_tp = sum(partials)   # one final all-reduce

print(f"Reference y_ref[0,0,:] = {y_ref[0,0,:]}")
print(f"TP reconstruction y_tp[0,0,:] = {y_tp[0,0,:]}")
print(f"Maximum error = {np.abs(y_ref - y_tp).max():.2e}")

print()
print("Key observation: each GPU computes its own attention heads, so attention itself needs no communication.")
print("TP partitions Attention elegantly at the natural boundary between heads.")


### 2.3 Communication Cost of Tensor Parallelism

TP reduces per-GPU weight memory linearly, but each Transformer layer communicates four times:

- the MLP has one forward and one backward all-reduce;
- Attention has one forward and one backward all-reduce.

Backward communication mirrors the forward reduction because gradients must be redistributed along the same partition. These reductions lie on the critical path. TP therefore normally uses fast intra-node NVLink and is usually limited to one node; slower inter-node scaling is left to PP or DP.


In [ ]:
# === Larger TP reduces per-GPU weights but not communication count ===
d_model, d_ff = 4096, 11008
bytes_elem = 2   # BF16

print(f"{'TP':>4}{'Per-GPU MLP weights (MB)':>20}{'all-reduces/layer':>20}")
print("-" * 46)
for tp in [1, 2, 4, 8]:
    w_bytes = (d_model * d_ff + d_ff * d_model) * bytes_elem / tp
    comms = 4 if tp > 1 else 0   # forward+backward × MLP+Attention; no communication at TP=1
    print(f"{tp:>4}{w_bytes / 1e6:>18.1f}{comms:>20}")

print()
print("Key observation: TP=8 cuts per-GPU weights to one eighth, but every layer still performs four all-reduces.")
print("Communication count does not fall as TP grows, so TP belongs within a fast-NVLink node.")


## 3. Pipeline Parallelism

Instead of partitioning one layer, Pipeline Parallelism (PP) assigns consecutive groups of layers to different GPUs. A naive schedule sends one batch through GPU 0, then GPU 1, and so on. Only one GPU works at a time, so memory is distributed but throughput does not improve.

### 3.1 Micro-Batches and Pipeline Scheduling

GPipe divides a batch into $M$ **micro-batches** and staggers them. As soon as stage 0 finishes micro-batch 0, it begins micro-batch 1 while stage 1 processes micro-batch 0. Most stages then work simultaneously.

Idle slots at the beginning and end are called the **pipeline bubble**. Its approximate fraction is

$$rac{P-1}{M},$$

where $P$ is the number of stages and $M$ the number of micro-batches. More micro-batches reduce the bubble.


In [ ]:
# === Hand calculation: bubble fraction = (P-1)/M ===
print(f"{'P (stages)':>10}{'M (microbatches)':>12}{'Bubble fraction':>14}")
print("-" * 38)
for P in [4, 8]:
    for M in [8, 32, 128]:
        bubble = (P - 1) / M
        print(f"{P:>10}{M:>12}{bubble*100:>12.1f}%")

print()
print("Key observation: with P=8 and M=128, the pipeline bubble is only 5.5%.")
print("In practice, M is usually at least four to eight times the stage count.")
print("M must exceed P-1; otherwise there are too few microbatches to fill the pipeline and the formula becomes absurd.")


### 3.2 Pipeline Timeline

The following character diagram makes staggering concrete. With four stages and four micro-batches, each cell is one time unit: `F0` and `B0` are the forward and backward passes of micro-batch 0, and `.` means idle. This simplified schedule illustrates GPipe; production systems also use variants such as 1F1B.


In [ ]:
# === Text pipeline timeline for P=4, M=4, simplified ===
P, M = 4, 4

def line(stage):
    """Build one stage's timeline: wait for fill, run forwards, then backwards."""
    cells = [" . "] * stage          # wait for earlier stages to start
    cells += [f"F{i}" for i in range(M)]   # process M forward microbatches
    cells += [" . "] * (P - 1 - stage)     # illustrative tail bubble
    cells += [f"B{i}" for i in range(M)]   # backward
    return cells

print("GPipe interleaved pipeline, P=4 and M=4; F=forward, B=backward, .=idle")
print("=" * 62)
for s in range(P):
    print(f"stage {s}: " + " ".join(line(s)))

print()
print("Key observation: later layers wait longer during the initial warmup bubble;")
print("Every stage has an idle cooldown bubble at the end.")
print("During most of the middle, every stage works simultaneously; that is the pipeline's value.")


### 3.3 1F1B and DualPipe

GPipe performs all forward passes at a stage before its backward passes, so it must retain many activations. Backpropagation needs intermediate forward results; a long delay therefore raises activation memory.

- **1F1B** alternates one forward and one backward operation at each stage, reducing retained activations while keeping the pipeline busy.
- **DualPipe**, introduced with DeepSeek-V3, sends half the micro-batches in each direction and interleaves them to reduce bubbles at both ends, at the cost of more memory.

PP therefore balances pipeline bubbles against activation memory.


## 4. Sequence Parallelism

Sometimes model weights fit but a very long input does not. Attention activations grow quadratically with sequence length because $QK^T$ has shape sequence length by sequence length. Sequence Parallelism (SP) partitions the sequence dimension.

It serves two purposes:

1. With TP, token-independent operations such as LayerNorm and dropout can operate on local sequence segments instead of replicating the entire sequence.
2. For very long contexts, methods such as Ring Attention exchange segments so each GPU can compute relationships between its local tokens and the other segments.

The next calculation shows the resulting memory reduction.


In [ ]:
# === SP effect on LayerNorm activation memory when partitioning the sequence ===
batch, seq_len, d_model = 1, 8192, 4096
bytes_elem = 2

full_bytes = batch * seq_len * d_model * bytes_elem
print(f"Full LayerNorm input activation: {full_bytes / 1e6:.1f} MB")
print(f"{'SP':>6}{'Per-GPU activation (MB)':>18}{'Saved':>10}")
print("-" * 36)
for sp in [1, 2, 4, 8]:
    per_card = full_bytes / sp
    print(f"{sp:>6}{per_card / 1e6:>16.1f}{(1 - 1/sp)*100:>8.0f}%")

print()
print("Key observation: SP=8 reduces per-GPU activation memory to one eighth.")
print("Cost: all-gather the sequence before TP and reduce-scatter it again afterward.")


## 5. Expert Parallelism

Expert Parallelism (EP) applies to Mixture-of-Experts models. An MoE layer replaces one FFN with many expert networks. A router sends each token to its top-$k$ experts, so total parameters can be large while each token activates only a small fraction.

When all expert weights do not fit on one GPU, EP assigns different experts to different GPUs. The experts themselves are the partitioned objects.

Tokens must travel to the GPUs that own their selected experts. An **all-to-all** dispatches tokens, and another all-to-all returns and combines the results. The following cell simulates dispatch.


In [ ]:
# === EP simulation: distribute four experts over four GPUs and inspect Token dispatch ===
import numpy as np

num_experts = 4
world_size = 4
seq_len = 8

d_model = 4
np.random.seed(1)
tokens = np.random.randn(seq_len, d_model)          # eight Tokens
router_logits = tokens @ np.random.randn(d_model, num_experts)
assignments = router_logits.argmax(axis=1)          # each Token selects one expert

print(f"Token -> expert routing: {assignments.tolist()}")

# Simulate all-to-all: each expert GPU gathers assigned Tokens
expert_inputs = [[] for _ in range(num_experts)]
for tok_idx, expert_id in enumerate(assignments):
    expert_inputs[expert_id].append(tokens[tok_idx])

for eid in range(num_experts):
    print(f"expert {eid} on GPU {eid} receives {len(expert_inputs[eid])} Tokens")

print()
print("Key observation: tokens travel from their current GPU to the GPU hosting the selected expert; this is all-to-all.")
print("After computation, results travel back along the reverse all-to-all path for combination.")


In [ ]:
# === Per-GPU expert-memory comparison for EP, TP, and DP in MoE ===
# Eight experts with 110M parameters each across eight GPUs
expert_params = 110e6
num_experts = 8
total_params = expert_params * num_experts
N = 8

ep_per_card = expert_params        # EP: one expert per GPU
pp_same = total_params / N          # if each expert were split eight ways
dp_per_card = total_params          # DP: complete copy on every GPU

print(f"{'Scheme':<10}{'Per-GPU expert weights (M)':>22}{'Communication':>16}")
print("-" * 50)
print(f"{'EP':<10}{ep_per_card/1e6:>20.1f}{'all-to-all':>16}")
print(f"{'TP':<10}{pp_same/1e6:>20.1f}{'all-reduce':>16}")
print(f"{'DP':<10}{dp_per_card/1e6:>20.1f}{'all-reduce (gradients)':>16}")

print()
print("Key observation: EP minimizes per-GPU storage but uses all-to-all, which cannot use ring reduction;")
print("EP therefore places the strongest demands on network topology, as emphasized in the hardware appendix.")


### 5.1 Communication Volume of Expert Parallelism

Each token is sent to its selected experts and returned afterward. Thus every MoE layer performs one dispatch and one combine all-to-all. The following Mixtral 8x7B estimate gives the order of magnitude.


In [ ]:
# === All-to-all traffic for one Mixtral 8x7B MoE layer ===
batch = 1024       # global batch
seq = 4096
hidden = 4096
top_k = 2
n_ep = 8           # EP degree
bytes_elem = 2     # BF16

# Tokens sent per GPU = batch×seq / n_ep × top_k, because each Token activates two experts
tokens_per_card = batch * seq / n_ep * top_k
bytes_per_send = tokens_per_card * hidden * bytes_elem

print(f"Mixtral 8x7B（EP={n_ep}, top_k={top_k}）：")
print(f"  One all-to-all send per GPU: {bytes_per_send / 1e9:.2f} GB")
print(f"  One MoE layer, dispatch plus combine: {2 * bytes_per_send / 1e9:.2f} GB")
print(f"  Total across 32 MoE layers: {32 * 2 * bytes_per_send / 1e9:.1f} GB")
print()
print("Key observation: MoE communication alone can move hundreds of GB in one training step;")
print("Even at EP=8, this is substantial. Larger EP lowers bytes per GPU but raises latency.")


## 6. Combining Five Parallelism Dimensions

Real large-model training combines several strategies because they partition different objects: data, weights, layers, sequences, and experts. A typical 2,048-GPU layout might use:

- an innermost group of eight GPUs for TP over NVLink;
- EP or PP across nearby nodes;
- DP as the outermost dimension to scale throughput.

The most communication-intensive dimension is placed on the fastest links. The following table compares three published configurations.


In [ ]:
# === Parallel configurations of three real models ===
# Total GPUs = TP × PP × DP × EP × SP
configs = [
    ("Llama-3 70B",  "dense", {"TP": 8,  "PP": 8,  "DP": 16, "EP": 1,  "SP": 1}),
    ("Mixtral 8x7B", "MoE",   {"TP": 1,  "PP": 1,  "DP": 16, "EP": 8,  "SP": 1}),
    ("DeepSeek-V3",  "MoE",   {"TP": 1,  "PP": 16, "DP": 2,  "EP": 64, "SP": 1}),
]

print(f"{'Model':<16}{'Type':<8}{'TP':>4}{'PP':>5}{'DP':>5}{'EP':>5}{'SP':>5}{'Total GPUs':>10}")
print("-" * 60)
for name, kind, c in configs:
    total = c["TP"] * c["PP"] * c["DP"] * c["EP"] * c["SP"]
    print(f"{name:<16}{kind:<8}{c['TP']:>4}{c['PP']:>5}{c['DP']:>5}{c['EP']:>5}{c['SP']:>5}{total:>10}")

print()
print("Interpretation:")
print("  Llama-3 is dense with no experts, using intra-node TP plus PP and DP.")
print("  Mixtral is MoE; EP=8 places its eight experts on separate GPUs, with remaining scaling from DP.")
print("  DeepSeek-V3 has 256 experts and relies on EP=64; PP=16 uses DualPipe scheduling.")


### 6.1 Principles for Combining Strategies

Three general rules explain the examples:

1. Keep TP within a node because it communicates frequently.
2. Use EP only for MoE, matching the expert count to the EP degree.
3. Let PP and DP consume the remaining GPUs, with enough micro-batches to hide pipeline bubbles.

Larger MoE models devote more parallel capacity to EP. Their all-to-all traffic is particularly sensitive to network topology and bandwidth.


## Summary

The reasoning chain is:

- A model does not fit, so partition it along one or more of five dimensions.
- DP partitions data but replicates model state; ZeRO partitions DP's redundant state.
- TP partitions weights inside a layer. A column-to-row MLP needs one forward all-reduce and normally stays within a node.
- PP partitions layers and uses staggered micro-batches; its bubble is approximately $(P-1)/M$.
- SP partitions sequences for long contexts; EP partitions experts and uses all-to-all.
- Total GPUs equal $TP	imes PP	imes DP	imes SP	imes EP$.

Check your understanding:

- [ ] DP multiplies throughput but does not reduce per-replica model memory.
- [ ] ZeRO progressively partitions optimizer states, gradients, and parameters.
- [ ] Column parallelism partitions outputs; row parallelism partitions inputs and needs an all-reduce.
- [ ] A column → activation → row MLP needs only one forward all-reduce.
- [ ] Attention heads are independent partition units.
- [ ] TP usually remains within an NVLink node.
- [ ] PP partitions layers, and more micro-batches reduce its bubble.
- [ ] SP partitions sequences; EP partitions experts and uses all-to-all.


## Exercises

> You may ask AI for hints, decomposition, or a direction check, but avoid asking it to complete the exercise.

**Exercise 1: Memory with TP=2 and DP=2**

A 7B dense model is trained with BF16 and AdamW on four GPUs using TP=2, DP=2, and no ZeRO. What does each GPU store, and how many GB of fixed state does it need?

Hint: TP halves weights and gradients, but optimizer states are not partitioned: $2P/2+2P/2+12P=14P$ bytes.


In [ ]:
# Exercise 1: per-GPU memory for a 7B model under TP=2 plus DP=2
P = 7e9

# TODO: total memory (GB)
per_card_bytes = None

assert per_card_bytes is not None, 'Please replace the placeholder before running the assertion.'
expected = 2 * P / 2 + 2 * P / 2 + 12 * P   # = 14P
assert abs(per_card_bytes - expected) < 1e9, f"Should be {expected / 1e9:.0f} GB"

print(f"Exercise 1 passed:")
print(f"  Per-GPU memory with TP=2 + DP=2 = {per_card_bytes / 1e9:.0f} GB")
print(f"  Compared with 112 GB for pure DDP, TP halves weights and gradients but not optimizer state.")


**Exercise 2: Compute the PP Bubble**

For PP=8 stages and $M=32$ micro-batches, what percentage of time is bubble? What does it become when $M=128$?

Hint: substitute directly into $(P-1)/M$.


In [ ]:
# Exercise 2: PP bubble fraction
P, M = 8, 32

# TODO: calculate bubble fraction as a number from 0 to 1
bubble_ratio = None

assert bubble_ratio is not None, 'Please replace the placeholder before running the assertion.'
expected = (P - 1) / M
assert abs(bubble_ratio - expected) < 1e-6, f"Should be {expected:.4f}"

print(f"Exercise 2 passed:")
print(f"  At PP={P}, M={M}, bubble fraction = {bubble_ratio*100:.1f}%")
print(f"  Raising M to 128 lowers the bubble to {(P-1)/128*100:.1f}%.")
print(f"  Engineering practice often uses microbatches = stages × 4 to 8 to suppress bubbles.")


**Exercise 3: Total GPUs for DeepSeek-V3**

For TP=1, PP=16, DP=2, EP=64, and SP=1, compute the total number of GPUs.

Hint: multiply all five parallel degrees.


In [ ]:
# Exercise 3: total GPU count for DeepSeek-V3
config = {"TP": 1, "PP": 16, "DP": 2, "EP": 64, "SP": 1}

# TODO: compute SwiGLU output
total_gpus = None

assert total_gpus is not None, 'Please replace the placeholder before running the assertion.'
expected = config["TP"] * config["PP"] * config["DP"] * config["EP"] * config["SP"]
assert total_gpus == expected, f"Should be {expected}"

print(f"Exercise 3 passed:")
print(f"  DeepSeek-V3 configuration: {config}")
print(f"  Total GPUs = {total_gpus}; the public report uses 2,048 H800s")
print(f"  EP=64 dominates: distribute 256 experts across 64 GPUs, four experts per GPU.")


## References

- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- Huang et al., [GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism](https://arxiv.org/abs/1811.06965), 2018
- Narayanan et al., [Memory-Efficient Pipeline-Parallel DNN Training (1F1B)](https://arxiv.org/abs/2004.13378), 2020
- DeepSeek-AI, [DeepSeek-V3 Technical Report (DualPipe)](https://arxiv.org/abs/2412.19437), 2024
- Korthikanti et al., [Reducing Activation Recomputation in Large Transformer Models (Sequence Parallelism)](https://arxiv.org/abs/2205.05198), 2022
- Fedus et al., [Switch Transformers (Expert Parallelism)](https://arxiv.org/abs/2101.03961), 2021
- Liu et al., [Ring Attention with Blockwise Transformers for Near-Infinite Context](https://arxiv.org/abs/2310.01889), 2023

